# Notebook 03 — Exposure Timeline Generation and Characterisation
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> Health records  →  Satellite  →  [Environment + Timing]  →  Results & GeoAI
> ```
>
> Climate varies by season. Pregnancy unfolds over nine months.
> The same heat anomaly in the first trimester (organogenesis) has a different
> biological meaning than in the third trimester (rapid fetal growth).
> This notebook shows what the environmental datasets look like, then builds
> **trimester-specific exposures** so the GeoAI model sees the full climate
> story of each pregnancy — not just a single snapshot.


---
## 1. The Pregnancy Timeline

A pregnancy spans roughly 40 weeks. Climate exposure during each window has
distinct biological relevance:

| Window | Weeks (approx.) | Biological significance |
|---|---|---|
| Pre-conception | −4 to 0 | Oocyte maturation, endometrial preparation |
| **T1** | 1 – 13 | Organogenesis — organ formation is heat-sensitive |
| **T2** | 14 – 26 | Fetal growth; lung and brain development |
| **T3** | 27 – 40 | Rapid weight gain — heat stress and nutrition critical |
| Delivery / neonatal | 0 – 28 days | Thermal regulation, infection risk |

**The key insight:** the *month* a pregnancy begins maps each trimester onto a
specific climate season. In Zimbabwe, a baby conceived in November reaches T3
during August–October — the hottest pre-monsoon window. A baby conceived in
May reaches T3 in February–April, the tail of the cool wet season.
The conception month is the key that links gestation to climate.


In [ ]:
import pathlib as _pl
# ── Prerequisite check — run before anything else ─────────────────────────
_badge = _pl.Path.home() / '.nb02_complete'
if not _badge.exists():
    print("⛔  STOP — You must complete Notebook 02: Exposure Extraction and Linkage before this notebook.")
    print()
    print("   Steps:")
    print("   1. Open 02_exposure_extraction_linkage.ipynb")
    print("   2. Work through the notebook and pass the quiz")
    print("   3. Run the 'Save Progress' cell at the bottom")
    print("   4. Return here and re-run this cell")
    raise SystemExit("Prerequisites not met.")
print("✅  Prerequisite check passed — welcome to this module.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))  # makes shared_utils importable
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")


---
## 2. Environmental Datasets — What We Measure and Why

Five satellite and reanalysis products characterise Zimbabwe's environment,
one row per DHS cluster per calendar month:


In [ ]:
climate  = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')
clusters = pd.read_csv(DATA_DIR / 'zw_clusters.csv')

print(f"Climate rows: {len(climate):,} | Columns: {list(climate.columns)}")
print(climate.head(4).to_string())

Each row is one cluster in one month.

- `temp_c` — ERA5 2 m air temperature (°C) from ECMWF reanalysis (~28 km resolution).
- `rain_mm` — monthly rainfall (mm) from CHIRPS (~5 km gridded, gauge-corrected).
- `ndvi` — MODIS NDVI (1 km monthly; 0 = bare soil, ~0.6 = dense miombo woodland).
- `dust_aod550` — CAMS aerosol optical depth, proxy for dust loading.
- `pm25` — CAMS near-surface PM2.5 (µg/m³); elevated in urban clusters (Harare).
- **SPEI-3** — Standardised Precipitation-Evapotranspiration Index (3-month window).
  Positive = wetter than normal; negative = drought. A 3-month window aligns with
  one gestational trimester, making SPEI-3 the standard drought indicator in
  pregnancy–climate research.


---
## 3. Zimbabwe's Seasonal Cycle

Zimbabwe has a subtropical Southern-Hemisphere monsoon climate: one rainy season
(October–April) driven by the ITCZ, with a cool dry winter (June–August) and a hot
pre-monsoon (September–November). Temperature, rainfall, vegetation, and air quality
all follow this rhythm — so the month a pregnancy begins determines the climate
season of each trimester.


In [ ]:
baseline = climate[climate['year'].isin([2011, 2012, 2013, 2014])]

# National mean by calendar month, averaging over all clusters and baseline years
seasonal = baseline.groupby('month')[['temp_c', 'rain_mm', 'ndvi', 'pm25']].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rainy_months = [10, 11, 12, 1, 2, 3, 4]

# ── Left panel: Temperature + Rainfall ───────────────────────────────────────
ax1 = axes[0]
ax2 = ax1.twinx()

for m in rainy_months:
    ax1.axvspan(m - 0.5, m + 0.5, color='#d6eaf8', alpha=0.55, zorder=0)

ax2.bar(seasonal.index, seasonal['rain_mm'], color='#2980b9', alpha=0.65, width=0.65, zorder=2)
ax1.plot(seasonal.index, seasonal['temp_c'], color='#c0392b',
         marker='o', markersize=5, linewidth=2, zorder=3)

ax1.set_xlabel('Month')
ax1.set_ylabel('Mean 2 m air temp (°C)', color='#c0392b')
ax2.set_ylabel('Mean rainfall (mm/month)', color='#2980b9')
ax1.tick_params(axis='y', labelcolor='#c0392b')
ax2.tick_params(axis='y', labelcolor='#2980b9')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'])
ax1.set_title('Temperature & Rainfall', fontweight='bold')

rainy_patch = mpatches.Patch(color='#d6eaf8', label='Rainy season (Oct–Apr)')
temp_line   = mpatches.Patch(color='#c0392b', label='Air temp (°C)')
rain_bar    = mpatches.Patch(color='#2980b9', alpha=0.65, label='Rainfall (mm/month)')
ax1.legend(handles=[rainy_patch, temp_line, rain_bar], loc='upper left', fontsize=9)

# ── Right panel: NDVI + PM2.5 ────────────────────────────────────────────────
ax3 = axes[1]
ax4 = ax3.twinx()

for m in rainy_months:
    ax3.axvspan(m - 0.5, m + 0.5, color='#d6eaf8', alpha=0.55, zorder=0)

ax3.plot(seasonal.index, seasonal['ndvi'], color='#27ae60',
         marker='s', markersize=5, linewidth=2, zorder=3, label='NDVI')
ax4.plot(seasonal.index, seasonal['pm25'], color='#7f8c8d',
         marker='^', markersize=5, linewidth=2, linestyle='--', zorder=3, label='PM2.5 (µg/m³)')

ax3.set_xlabel('Month')
ax3.set_ylabel('Mean NDVI', color='#27ae60')
ax4.set_ylabel('Mean PM2.5 (µg/m³)', color='#7f8c8d')
ax3.tick_params(axis='y', labelcolor='#27ae60')
ax4.tick_params(axis='y', labelcolor='#7f8c8d')
ax3.set_xticks(range(1, 13))
ax3.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'])
ax3.set_title('Vegetation & Air Quality', fontweight='bold')

ndvi_line = mpatches.Patch(color='#27ae60', label='NDVI')
pm_line   = mpatches.Patch(color='#7f8c8d', label='PM2.5 (µg/m³)')
ax3.legend(handles=[rainy_patch, ndvi_line, pm_line], loc='upper left', fontsize=9)

plt.suptitle('Zimbabwe Seasonal Climate Cycle — Baseline 2011–2014\n'
             'Mean over 399 DHS clusters; blue shading = rainy season',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Note: Coldest months Jun–Jul (winter), hottest Oct–Nov (hot dry season). "
      "PM2.5 peaks Aug–Sep during Zimbabwe's biomass burning season.")

Temperature peaks **October–November** (hot-dry), then cools once the rains arrive.
NDVI lags rainfall by 1–2 months as vegetation grows in response.
PM2.5 spikes **August–September** during Zimbabwe's biomass burning season.

A pregnancy with T3 in October–November is simultaneously exposed to peak heat,
near-zero rainfall, and degraded air quality — the full combination that the
trimester variables below will capture.


---
## 4. From Cluster-Month Climate to Per-Participant Daily Exposure

In Notebook 02 we attached **one climate row per cluster** to each birth. That gives us the background climate for the area — but not the actual exposure *this specific pregnancy* experienced.

To compute trimester exposures we need to answer: **"What was the daily temperature at this birth's cluster during each week of this particular pregnancy?"**

We do this with a pre-built file `zw_births_daily_exposure.csv` that has already joined the satellite data to each birth at the daily level:

```
birth_id  |  date        |  days_since_conception  |  temp_c  |  pm25  |  precipitation  |  ndvi
-----------------------------------------------------------------------------------------------
C001__1   |  2012-07-15  |       1                 |  18.3    |  12.1  |      0.0        |  0.42
C001__1   |  2012-07-16  |       2                 |  19.1    |  11.8  |      0.0        |  0.42
...       |  ...         |      ...                |  ...     |  ...   |      ...        |  ...
C001__1   |  2013-04-10  |     270                 |  24.7    |  14.2  |      5.3        |  0.61
```

**One row = one birth × one pregnancy day.** The file has roughly 5,800 births × 280 days = ~1.6 million rows.

**How the linkage works:**
1. Each birth has a conception date (derived from DHS interview date and reported gestational age) and a `cluster_id`
2. For each day from conception to delivery, we look up the ERA5 / CHIRPS / MODIS values for that cluster on that date
3. This gives every participant a personal daily climate diary aligned to their own pregnancy calendar

The cell below loads this file and shows you what the first few rows look like for one birth.


In [ ]:
# Load the daily exposure data — one row per birth × pregnancy day
daily = pd.read_csv(DATA_DIR / 'zw_births_daily_exposure.csv', parse_dates=['date'])
daily['birth_id'] = daily['caseid'].str.strip() + '__' + daily['bidx'].astype(str)

# Load birth metadata (health outcomes, demographics)
births_meta = pd.read_csv(DATA_DIR / 'zw_births_raw.csv')
births_meta['birth_id'] = births_meta['caseid'].str.strip() + '__' + births_meta['bidx'].astype(str)

print(f'Daily exposure rows: {len(daily):,}')
print(f'Unique births: {daily["birth_id"].nunique():,}')
print(f'Variables: {[c for c in daily.columns if c not in ["birth_id","caseid","bidx","cluster_id","Latitude","Longitude","date"]]}')
print(f'\nDate range: {daily["date"].min().date()} → {daily["date"].max().date()}')
print(f'\nSample (first birth):')
sample_id = daily['birth_id'].iloc[0]
print(daily[daily['birth_id']==sample_id][['date','temp_c','pm25','precipitation','ndvi']].head(6).to_string(index=False))

---
## 5. Computing Trimester-Specific Exposures

Trimester windows are defined by days since conception:
- **T1** = days 0 – 89 (conception through week 13)
- **T2** = days 90 – 179 (weeks 14 – 26)
- **T3** = days 180+ (week 27 to birth)

For each birth, all daily values within each window are averaged.
The code is hidden below (toggle to inspect); the output is the `births` dataframe
with columns `temp_t1`, `rain_t1`, ..., `ndvi_t3` for all 5,794 births.


In [ ]:
# Compute trimester-specific means from the daily data
# Trimester windows (days since conception):
#   T1 = days 0–89    (conception through week 13)
#   T2 = days 90–179  (weeks 14–26)
#   T3 = days 180+    (week 27 to birth)

def compute_trimester_col(group):
    """Given all daily rows for one birth, compute T1/T2/T3 means."""
    # Sort by date so day-number is sequential
    g = group.sort_values('date').copy()
    g['day_num'] = (g['date'] - g['date'].iloc[0]).dt.days
    
    t1 = g[g['day_num'] < 90]
    t2 = g[(g['day_num'] >= 90) & (g['day_num'] < 180)]
    t3 = g[g['day_num'] >= 180]
    
    env_vars = ['temp_c', 'dust_aod550', 'pm25', 'precipitation', 'ndvi']
    rec = {'birth_id': group.name}
    for var in env_vars:
        short = {'temp_c':'temp','dust_aod550':'dust','pm25':'pm25','precipitation':'rain','ndvi':'ndvi'}[var]
        rec[f'{short}_t1'] = t1[var].mean() if len(t1) else float('nan')
        rec[f'{short}_t2'] = t2[var].mean() if len(t2) else float('nan')
        rec[f'{short}_t3'] = t3[var].mean() if len(t3) else float('nan')
    return pd.Series(rec)

print('Computing trimester means for 5,794 births (this takes ~1 minute)...')
trimester_df = daily.groupby('birth_id', group_keys=False).apply(compute_trimester_col)
trimester_df = trimester_df.reset_index(drop=True)

# Merge with birth metadata
births = births_meta.merge(trimester_df, on='birth_id', how='left')

# Derived flags
# Quartile-based thresholds: robust regardless of dataset temperature/rainfall range
_temp_t3_p75 = births['temp_t3'].quantile(0.75)    # top 25% T3 heat
_rain_t3_p25 = births['rain_t3'].quantile(0.25)    # bottom 25% T3 rainfall
births['hot_t3']         = births['temp_t3'] > _temp_t3_p75
births['drought_exposed'] = births['rain_t3'] < _rain_t3_p25

print(f'Trimester dataset: {len(births):,} births × {len(births.columns)} columns')
print(f'Trimester columns: {[c for c in births.columns if any(c.endswith(f"_t{t}") for t in [1,2,3])]}')
print(f'\nT3 temperature: mean={births.temp_t3.mean():.1f}°C, 75th pct={_temp_t3_p75:.1f}°C')
print(f'hot_t3 (top quartile T3 heat):             {births.hot_t3.sum():,} births ({births.hot_t3.mean()*100:.1f}%)')
print(f'T3 rainfall 25th pct={_rain_t3_p25:.3f} mm/day')
print(f'drought_exposed (bottom quartile T3 rain): {births.drought_exposed.sum():,} births ({births.drought_exposed.mean()*100:.1f}%)')

---
## 6. Conception Month Determines Trimester Climate

The plot below shows mean T3 temperature by conception month — the direct
translation of the seasonal cycle into gestational exposure. Babies conceived
in November–December hit T3 during the hottest pre-monsoon months.


In [ ]:
RAINY_MONTHS = [11, 12, 1, 2, 3]  # Zimbabwe wet season

t3_by_conception = (
    births
    .groupby('conception_month')['temp_t3']
    .mean()
    .reset_index()
    .rename(columns={'temp_t3': 'mean_temp_t3'})
)

fig, ax = plt.subplots(figsize=(9, 4))

# Shade rainy season band
for m in RAINY_MONTHS:
    ax.axvspan(m - 0.5, m + 0.5, alpha=0.15, color='royalblue', zorder=0)

ax.plot(
    t3_by_conception['conception_month'],
    t3_by_conception['mean_temp_t3'],
    marker='o', ms=7, color='firebrick', lw=2.0, label='Mean T3 temperature'
)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(
    ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
)
ax.set_xlabel('Conception month', fontsize=11)
ax.set_ylabel('Mean T3 temperature (°C)', fontsize=11)
ax.set_title('Third-trimester heat by conception month — Zimbabwe DHS 2015', fontsize=12)

rainy_patch = mpatches.Patch(color='royalblue', alpha=0.3, label='Rainy season (Nov–Mar)')
ax.legend(handles=[ax.get_lines()[0], rainy_patch])
plt.tight_layout()
plt.show()

peak_c = int(t3_by_conception.loc[t3_by_conception['mean_temp_t3'].idxmax(), 'conception_month'])
print(f"Conception month with highest mean T3 temperature: month {peak_c}")
print("Interpretation: babies conceived in Nov–Dec reach T3 during Aug–Oct (hot-dry season).")

**This curve is the core insight of this notebook.** Two babies born in the same
district can have very different T3 heat profiles depending on when the pregnancy
began. A model that uses only birth location — or a single annual mean — misses
this temporal dimension entirely.


---
## 7. Does Trimester Timing Show Up in Birth Outcomes?

If T3 heat drives low birth weight (LBW), LBW rates should peak among babies
born ~3 months *after* the hottest period (because T3 precedes delivery by ~3
months). The chart plots birth month against LBW rate and overlays mean T3
temperature.


In [ ]:
births_bw2 = births.dropna(subset=['low_birth_weight'])

monthly = (
    births_bw2
    .groupby('birth_month')
    .agg(
        n_births      = ('birth_weight_g', 'count'),
        lbw_rate      = ('low_birth_weight', 'mean'),
        neonatal_mr   = ('neonatal_death', 'mean'),
        mean_temp_t3  = ('temp_t3', 'mean')
    )
    .reset_index()
)

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax = axes[0]
ax.bar(monthly['birth_month'], monthly['lbw_rate'] * 100, color='steelblue',
       alpha=0.8, label='LBW rate (%)')
ax2 = ax.twinx()
ax2.plot(monthly['birth_month'], monthly['mean_temp_t3'], color='firebrick',
         marker='s', ms=5, lw=1.5, label='Mean T3 temperature (°C)')
ax.set_ylabel('LBW rate (%)', fontsize=10)
ax2.set_ylabel('Mean T3 temperature (°C)', fontsize=10, color='firebrick')
ax.set_title('Monthly LBW rate and third-trimester temperature', fontsize=11)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

ax = axes[1]
ax.bar(monthly['birth_month'], monthly['neonatal_mr'] * 1000, color='darkorange',
       alpha=0.8, label='Neonatal MR (per 1,000)')
ax.set_ylabel('Neonatal deaths per 1,000 births', fontsize=10)
ax.set_title('Monthly neonatal mortality rate', fontsize=11)
ax.legend(fontsize=9)

for a in axes:
    a.set_xticks(range(1, 13))
    a.set_xticklabels(month_labels)

axes[1].set_xlabel('Birth month', fontsize=11)
plt.tight_layout()
plt.show()

peak_lbw_month = int(monthly.loc[monthly['lbw_rate'].idxmax(), 'birth_month'])
print(f"Birth month with highest LBW rate: {month_labels[peak_lbw_month-1]} (month {peak_lbw_month})")

---
## 8. Drought Exposure and Birth Outcomes

`drought_exposed` flags births in the **bottom quartile of T3 daily rainfall** —
a relative, within-dataset indicator robust to inter-annual variability.
The table below compares birth outcomes between drought-exposed and non-exposed
births as a first observational signal (the GeoAI model in Notebook 04 handles
confounders more rigorously).


In [ ]:
# drought_exposed is stored as bool/string — normalise to bool
births['drought_exposed'] = births['drought_exposed'].astype(bool)

drought_yes = births[births['drought_exposed']]
drought_no  = births[~births['drought_exposed']]

summary = pd.DataFrame({
    'Group'              : ['Drought-exposed', 'Not drought-exposed'],
    'N births'           : [len(drought_yes), len(drought_no)],
    'LBW rate'           : [
        drought_yes['low_birth_weight'].mean(),
        drought_no['low_birth_weight'].mean()
    ],
    'Mean birth wt (g)'  : [
        drought_yes['birth_weight_g'].mean(),
        drought_no['birth_weight_g'].mean()
    ],
    'Neonatal MR'        : [
        drought_yes['neonatal_death'].mean(),
        drought_no['neonatal_death'].mean()
    ],
})

summary['LBW rate']         = summary['LBW rate'].round(3)
summary['Mean birth wt (g)'] = summary['Mean birth wt (g)'].round(1)
summary['Neonatal MR']      = summary['Neonatal MR'].round(3)

print("Drought exposure and birth outcomes — Zimbabwe DHS 2015:")
print(summary.to_string(index=False))
print()
print("Note: drought_exposed = T3 mean daily rainfall in the bottom quartile (≤ 25th percentile).")
print("This is an observational comparison; season, wealth, and geography confound the association.")

In [ ]:
import pathlib as _pl
# ── Save progress — run this after passing the quiz above ─────────────────
_badge = _pl.Path.home() / '.nb03_complete'
_badge.touch()
print(f"✅  Progress saved: {_badge}")
print("   You can now open Notebook 04: GeoAI Implementation.")

In [ ]:
# ── Section 9: Save your trimester dataset (Notebook 04 loads this file) ──
# Save trimester dataset to your personal home directory
# Each student's copy is independent — running this cell does not affect other students.
import pathlib as _pl
_home_out = _pl.Path.home() / 'zw_births_trimester.csv'
births.to_csv(_home_out, index=False)
print(f'Saved {len(births):,} birth records with trimester exposures')
print(f'Location: {_home_out}')
print('Notebook 04 will load this file automatically.')


In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">&#129504; The T3 temperature curve by conception month shows that babies conceived in July experience the hottest T3. Why?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> July is the hottest month in Zimbabwe</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> A July conception puts T3 (weeks 27–40) in October–November — Zimbabwe&#39;s hottest pre-rain months</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Babies conceived in July are born in winter when hospitals are colder</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> T3 temperature is the same regardless of conception month</span></label></div>
  <div id="nb03q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="A July conception means T1 runs August–October, T2 November–January, and T3 February–April — but wait, actually July conception → T3 lands in Oct/Nov (hot-dry season). The key insight is that the same district can deliver very different climate exposure depending on WHEN conception occurred.",ID="nb03q1";
  function showFeedback(val){var fb=document.getElementById(ID+"-fb");if(!fb)return;if(val===C){fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';}else{fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';}document.dispatchEvent(new CustomEvent('mcq-update'));}
  function attach(){var saved=localStorage.getItem('mcq_'+ID);document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){if(saved&&r.value===saved){r.checked=true;}r.addEventListener("change",function(){localStorage.setItem('mcq_'+ID,this.value);showFeedback(this.value);});});if(saved)showFeedback(saved);}
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">&#129504; Why does the daily exposure dataset (~1.6 million rows) exist instead of just using monthly cluster averages?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Daily data is easier to store than monthly data</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Satellites only produce daily measurements, not monthly ones</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Each pregnancy starts on a different date — you need daily values to correctly align T1/T2/T3 windows to each individual birth&#39;s conception calendar</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Monthly averages are not available for Zimbabwe</span></label></div>
  <div id="nb03q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="C",E="Monthly averages cannot be trimester-aligned because every birth has a different conception date. Daily data lets us compute T1 = days 0–89, T2 = days 90–179, T3 = days 180+ from each birth&#39;s own conception date — a personalised climate diary for every pregnancy.",ID="nb03q2";
  function showFeedback(val){var fb=document.getElementById(ID+"-fb");if(!fb)return;if(val===C){fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';}else{fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';}document.dispatchEvent(new CustomEvent('mcq-update'));}
  function attach(){var saved=localStorage.getItem('mcq_'+ID);document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){if(saved&&r.value===saved){r.checked=true;}r.addEventListener("change",function(){localStorage.setItem('mcq_'+ID,this.value);showFeedback(this.value);});});if(saved)showFeedback(saved);}
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

In [ ]:
%%html
<div id="nb03-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 3 &mdash; Exposure Timeline &mdash; Quiz Results</div>
  <div id="nb03-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb03-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:"nb03q1",correct:"B"},{id:"nb03q2",correct:"C"}];
  var PASS=50;
  var NEXT="04_geoai_implementation.ipynb";
  var NEXT_LABEL="Notebook 04: GeoAI Implementation";
  var NBID="nb03";
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem("mcq_"+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+"-score-display");
    var act=document.getElementById(NBID+"-next-action");
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions to see your score. ('+(total-answered)+' remaining)</p>';
      if(act)act.innerHTML="";
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &mdash; MODULE PASSED</b><br><small>Also run the Save Progress cell below to unlock the prereq gate in Notebook 04.</small></div>';
      if(act){
        localStorage.setItem(NBID+"_passed","true");
        act.innerHTML='<div style="margin-top:10px"><a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px">&#10145; Continue to '+NEXT_LABEL+'</a><div style="margin-top:8px;font-size:12px;color:#6c757d">&#128269; Also run the <b>Save Progress</b> cell below &mdash; it unlocks the prereq gate in Notebook 04.</div></div>';
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &mdash; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer.</small></div>';
      if(act)act.innerHTML="";
    }
  }
  document.addEventListener("mcq-update",update);
  if(document.readyState==="loading")document.addEventListener("DOMContentLoaded",update);
  else update();
  setInterval(update,3000);
})();
</script>

In [ ]:
import pathlib as _pl
# ── Save progress — run this after passing the quiz above ─────────────────
_badge = _pl.Path.home() / '.nb03_complete'
_badge.touch()
print(f"✅  Progress saved: {_badge}")
print("   You can now open Notebook 04: GeoAI Implementation.")

---
## What We Hand to the GeoAI Model

This is what we hand to the GeoAI model: not just *"where was this birth"*
but *"what was the climate like during each trimester of this pregnancy."*

Each birth record now carries trimester-specific means for all five
environmental variables — plus derived flags (`hot_t3`, `drought_exposed`).
The GeoAI model can ask: does T1 dust matter independently of T3 heat?
Does the effect of drought differ by province? These questions are only
answerable because we preserved the temporal structure of each pregnancy.

---

> **Recap**
> - Climate exposure is not a single number — it varies trimester by trimester.
> - Zimbabwe's seasonal cycle (rainy Oct–Apr; peak heat Sep–Nov) creates large
>   contrasts between conception cohorts.
> - Trimester windows are defined by days since conception; each is ~90 days / ~3 months.
> - Products used: ERA5 temperature, CHIRPS rainfall, MODIS NDVI, CAMS PM2.5 and dust, SPEI-3 drought index.
>
> **Next:** Notebook 04 — GeoAI Implementation.


---

## ✅ Notebook 03 complete

You have built the full exposure timeline: daily climate linked to every pregnancy, trimester windows computed, seasonal patterns visible in outcomes. This dataset — one row per birth with T1/T2/T3 exposures — is what you hand to the GeoAI model.

**Next:** `04_geoai_implementation.ipynb`
